# Incremental Ingestion (`ingest_nbk`)

Inject **new chunks + their precomputed embeddings** (already processed upstream, e.g. on HPC)
into the `optimized_*` family, then refresh the Vector Search index.

The upsert is idempotent (MERGE on `chunk_id`), so re-running with overlapping ids updates rows
in place rather than duplicating them.


In [ ]:
# --- Make scripts/ importable ---
# In Databricks Repos the repo root is on sys.path; locate the scripts/ subdir there
# (works the same when running from the repo root locally).
import os, sys

_cands = [os.path.join(p, "scripts") for p in (list(sys.path) + [os.getcwd()])]
for _cand in _cands:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import settings
from ingest import load_new, ingest, sync_vector_index, rebuild_bm25
from lake_io import read_chunks_jsonl, read_embeddings_npy

## 1. Point at the new (HPC) output

The new data is expected in the same layout as the lake:
```
{NEW_BASE}/optimized_chunks/*/chunks.jsonl
{NEW_BASE}/optimized_embeddings/{model}/*/embeddings.npy
{NEW_BASE}/optimized_embeddings/{model}/*/chunk_ids.json
```


In [ ]:
# Location of the newly processed batch (edit per run)
NEW_BASE = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files_incoming/"

chunks_df, emb_df = load_new(spark, NEW_BASE)
print("new chunks    :", chunks_df.count())
print("new embeddings:", emb_df.count())
display(chunks_df.limit(5))

## 2. Ingest (upsert) + sync the Vector Search index

`sync_index=True` triggers the Delta Sync index to pick up the new rows. This needs
`settings.VS_ENDPOINT` (or the `TDIS_VS_ENDPOINT` env var) to be set.

In [ ]:
report = ingest(spark, chunks_df, emb_df, sync_index=True, wait_for_sync=True)
print(report)

## 3. (Optional) Refresh BM25 for keyword recall

Ingestion does not touch the BM25 tables. Rebuild them so keyword search also sees the new chunks.

In [ ]:
print(rebuild_bm25(spark))